# PJM RTO Load Ramp Analysis

Hourly ramp (MW delta) analysis for DA load forecasts — **RTO region only**.

- **Ramp** = hour-over-hour change in forecasted load (MW)
- Uses the latest forecast execution from today's DA window
- Compares forecasted ramps vs actual RT ramps for the previous 3 days

## 1. Setup & Configuration

In [1]:
import sys
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Repo root & backend imports ──
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))

from src.utils.azure_postgresql import pull_from_db

# ── Logger ──
try:
    from src.utils.logging_utils import init_logging
    pl = init_logging(name="ramp_analysis", log_to_file=False, level=logging.INFO)
    logger = pl.logger
except Exception:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    logger = logging.getLogger("ramp_analysis")

# ── Configurable Parameters ──
# Change these to re-run for any target date
TARGET_DATE = pd.Timestamp.now().normalize()  # today
FORECAST_DATE = TARGET_DATE + pd.Timedelta(days=1)  # tomorrow
HISTORY_DAYS = 3  # days of history for validation & RT comparison
MAX_EXECUTION_HOUR = 10  # latest forecast execution hour to include

logger.info(f"Target date (execution): {TARGET_DATE.date()}")
logger.info(f"Forecast date (tomorrow): {FORECAST_DATE.date()}")
logger.info(f"History window: {HISTORY_DAYS} days")

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src
INFO:ramp_analysis:Target date (execution): 2026-03-11
INFO:ramp_analysis:Forecast date (tomorrow): 2026-03-12
INFO:ramp_analysis:History window: 3 days


## 2. Data Pull — DA Load Forecast + RT Actuals

In [2]:
# ── DA Load Forecast: tomorrow's forecast from latest execution today (RTO) ──
sql_path = Path.cwd() / "ramp_da_forecast.sql"
query_da_forecast = sql_path.read_text().format(
    target_date=TARGET_DATE.date(),
    forecast_date=FORECAST_DATE.date(),
    max_execution_hour=MAX_EXECUTION_HOUR,
)

df_da = pull_from_db(query=query_da_forecast)
logger.info(f"DA forecast: {len(df_da):,} rows")
logger.info(f"Forecast date: {df_da['forecast_date'].iloc[0]} | Execution: {df_da['forecast_execution_datetime'].iloc[0]}")
df_da.head(10)

INFO:ramp_analysis:DA forecast: 24 rows
INFO:ramp_analysis:Forecast date: 2026-03-12 | Execution: 2026-03-11 08:47:26


,forecast_rank,forecast_execution_datetime,forecast_execution_date,forecast_datetime,forecast_date,hour_ending,region,forecast_load_mw
0,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 00:00:00,2026-03-12,1,RTO,78523.0
1,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 01:00:00,2026-03-12,2,RTO,76532.0
2,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 02:00:00,2026-03-12,3,RTO,75307.0
3,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 03:00:00,2026-03-12,4,RTO,74932.0
4,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 04:00:00,2026-03-12,5,RTO,76394.0
5,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 05:00:00,2026-03-12,6,RTO,80223.0
6,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 06:00:00,2026-03-12,7,RTO,86766.0
7,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 07:00:00,2026-03-12,8,RTO,91362.0
8,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 08:00:00,2026-03-12,9,RTO,93197.0
9,1,2026-03-11 08:47:26,2026-03-11,2026-03-12 09:00:00,2026-03-12,10,RTO,93261.0


In [3]:
# ── RT Metered Actuals: previous 3 days for ramp comparison (RTO) ──
history_start = (TARGET_DATE - pd.Timedelta(days=HISTORY_DAYS)).date()
history_end = (TARGET_DATE - pd.Timedelta(days=1)).date()

sql_path = Path.cwd() / "ramp_rt_actuals.sql"
query_rt_actuals = sql_path.read_text().format(
    history_start=history_start,
    history_end=history_end,
)

df_rt = pull_from_db(query=query_rt_actuals)

if df_rt is not None and len(df_rt) > 0:
    RT_AVAILABLE = True
    logger.info(f"RT actuals: {len(df_rt):,} rows | Date range: {df_rt['date'].min()} to {df_rt['date'].max()}")
else:
    RT_AVAILABLE = False
    logger.warning("No RT metered data available for the history window")

# ── DA Forecast for history window (for forecast vs actual ramp comparison) ──
sql_path = Path.cwd() / "ramp_da_history.sql"
query_da_history = sql_path.read_text().format(
    history_start=history_start,
    history_end=history_end,
    max_execution_hour=MAX_EXECUTION_HOUR,
)

df_da_hist = pull_from_db(query=query_da_history)
if df_da_hist is not None and len(df_da_hist) > 0:
    logger.info(f"DA history: {len(df_da_hist):,} rows | Dates: {df_da_hist['forecast_date'].min()} to {df_da_hist['forecast_date'].max()}")
else:
    logger.warning("No DA forecast history available")

INFO:ramp_analysis:RT actuals: 47 rows | Date range: 2026-03-08 to 2026-03-09
INFO:ramp_analysis:DA history: 71 rows | Dates: 2026-03-08 to 2026-03-10


## 3. Data Validation — Previous 3 Days (RTO)

Check completeness, nulls, outliers, temporal continuity for DA forecast and RT metered load.

In [4]:
validation_results = []

# ── DA Forecast Validation ──
print("=" * 70)
print("DA LOAD FORECAST (RTO) — Previous 3 Days")
print("=" * 70)

if df_da_hist is not None and len(df_da_hist) > 0:
    for date in sorted(df_da_hist["forecast_date"].unique()):
        dfr = df_da_hist[df_da_hist["forecast_date"] == date].sort_values("hour_ending")
        n_hours = dfr["hour_ending"].nunique()
        nulls = dfr["forecast_load_mw"].isna().sum()
        load_min = dfr["forecast_load_mw"].min()
        load_max = dfr["forecast_load_mw"].max()
        load_mean = dfr["forecast_load_mw"].mean()
        expected_hours = set(range(1, 25))
        actual_hours = set(dfr["hour_ending"].astype(int))
        missing_hours = expected_hours - actual_hours
        std = dfr["forecast_load_mw"].std()
        outliers = ((dfr["forecast_load_mw"] - load_mean).abs() > 3 * std).sum() if std > 0 else 0
        passed = n_hours == 24 and nulls == 0 and len(missing_hours) == 0

        validation_results.append({
            "source": "DA Forecast", "date": date,
            "hours": n_hours, "nulls": nulls, "missing_hours": len(missing_hours),
            "min_mw": round(load_min, 0), "max_mw": round(load_max, 0),
            "mean_mw": round(load_mean, 0), "outliers": outliers, "pass": passed,
        })

    df_val_da = pd.DataFrame(validation_results)
    display(df_val_da.style.set_caption("DA Forecast Validation (RTO)").map(
        lambda v: "background-color: #2d5a2d" if v is True else ("background-color: #5a2d2d" if v is False else ""),
        subset=["pass"],
    ))
else:
    print("No DA forecast history data to validate.")

# ── RT Metered Validation ──
print("\n" + "=" * 70)
print("RT METERED LOAD (RTO) — Previous 3 Days")
print("=" * 70)

rt_validation = []
if RT_AVAILABLE:
    for date in sorted(df_rt["date"].unique()):
        dfr = df_rt[df_rt["date"] == date].sort_values("hour_ending")
        n_hours = dfr["hour_ending"].nunique()
        nulls = dfr["rt_load_mw"].isna().sum()
        load_min = dfr["rt_load_mw"].min()
        load_max = dfr["rt_load_mw"].max()
        load_mean = dfr["rt_load_mw"].mean()
        expected_hours = set(range(1, 25))
        actual_hours = set(dfr["hour_ending"].astype(int))
        missing_hours = expected_hours - actual_hours
        std = dfr["rt_load_mw"].std()
        outliers = ((dfr["rt_load_mw"] - load_mean).abs() > 3 * std).sum() if std > 0 else 0
        passed = n_hours == 24 and nulls == 0 and len(missing_hours) == 0

        rt_validation.append({
            "source": "RT Metered", "date": date,
            "hours": n_hours, "nulls": nulls, "missing_hours": len(missing_hours),
            "min_mw": round(load_min, 0), "max_mw": round(load_max, 0),
            "mean_mw": round(load_mean, 0), "outliers": outliers, "pass": passed,
        })

    df_val_rt = pd.DataFrame(rt_validation)
    display(df_val_rt.style.set_caption("RT Metered Validation (RTO)").map(
        lambda v: "background-color: #2d5a2d" if v is True else ("background-color: #5a2d2d" if v is False else ""),
        subset=["pass"],
    ))
else:
    print("No RT metered data available to validate.")

# ── Pass/Fail Summary ──
all_validation = validation_results + rt_validation
if all_validation:
    total = len(all_validation)
    passed = sum(1 for v in all_validation if v["pass"])
    failed = total - passed
    print(f"\n{'=' * 70}")
    print(f"VALIDATION SUMMARY: {passed}/{total} passed, {failed}/{total} failed")
    print(f"{'=' * 70}")

DA LOAD FORECAST (RTO) — Previous 3 Days


,source,date,hours,nulls,missing_hours,min_mw,max_mw,mean_mw,outliers,pass
0,DA Forecast,2026-03-08,23,0,1,68152.000000,86947.000000,77509.000000,0,False
1,DA Forecast,2026-03-09,24,0,0,74108.000000,93564.000000,84790.000000,0,True
2,DA Forecast,2026-03-10,24,0,0,72524.000000,94453.000000,84817.000000,0,True



RT METERED LOAD (RTO) — Previous 3 Days


,source,date,hours,nulls,missing_hours,min_mw,max_mw,mean_mw,outliers,pass
0,RT Metered,2026-03-08,23,0,1,72076.000000,87902.000000,78795.000000,0,False
1,RT Metered,2026-03-09,24,0,0,73535.000000,93052.000000,83718.000000,0,True



VALIDATION SUMMARY: 3/5 passed, 2/5 failed


## 4. Compute Ramp Metrics

Hour-over-hour load delta: `ramp_mw = forecast_load_mw[HE] - forecast_load_mw[HE-1]`

In [5]:
# ── DA Forecast Ramps (tomorrow, RTO) ──
df_da = df_da.sort_values("hour_ending").reset_index(drop=True)
df_da["ramp_mw"] = df_da["forecast_load_mw"].diff()

logger.info(f"Computed DA ramps: {len(df_da)} rows")
logger.info(f"HE1 NaN (expected 1): {df_da['ramp_mw'].isna().sum()}")

# ── RT Actual Ramps (previous 3 days, RTO) ──
if RT_AVAILABLE:
    df_rt = df_rt.sort_values(["date", "hour_ending"]).reset_index(drop=True)
    df_rt["hour_ending"] = df_rt["hour_ending"].astype(int)
    df_rt["ramp_mw"] = df_rt.groupby("date")["rt_load_mw"].diff()
    logger.info(f"Computed RT ramps: {len(df_rt)} rows")

# ── DA Forecast Ramps (history, for comparison) ──
if df_da_hist is not None and len(df_da_hist) > 0:
    df_da_hist = df_da_hist.sort_values(["forecast_date", "hour_ending"]).reset_index(drop=True)
    df_da_hist["ramp_mw"] = df_da_hist.groupby("forecast_date")["forecast_load_mw"].diff()
    logger.info(f"Computed DA history ramps: {len(df_da_hist)} rows")

# Preview
df_da[["hour_ending", "forecast_load_mw", "ramp_mw"]].head(10)

INFO:ramp_analysis:Computed DA ramps: 24 rows
INFO:ramp_analysis:HE1 NaN (expected 1): 1
INFO:ramp_analysis:Computed RT ramps: 47 rows
INFO:ramp_analysis:Computed DA history ramps: 71 rows


,hour_ending,forecast_load_mw,ramp_mw
0,1,78523.0,NaN
1,2,76532.0,-1991.0
2,3,75307.0,-1225.0
3,4,74932.0,-375.0
4,5,76394.0,1462.0
5,6,80223.0,3829.0
6,7,86766.0,6543.0
7,8,91362.0,4596.0
8,9,93197.0,1835.0
9,10,93261.0,64.0


## 5. Ramp Bar Chart — RTO Hourly MW Delta

Positive ramps (load increasing) in green, negative ramps (load decreasing) in red.

In [6]:
dfr = df_da.dropna(subset=["ramp_mw"]).copy()
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in dfr["ramp_mw"]]

fig = go.Figure(go.Bar(
    x=dfr["hour_ending"],
    y=dfr["ramp_mw"],
    marker_color=colors,
    text=[f"{v:+,.0f}" for v in dfr["ramp_mw"]],
    textposition="outside",
    textfont=dict(size=9),
    hovertemplate="<b>RTO</b><br>HE%{x}<br>Ramp: %{y:+,.0f} MW<extra></extra>",
))
fig.add_hline(y=0, line_color="gray", line_width=0.5)
fig.update_layout(
    title=f"RTO — DA Load Forecast Ramps — {FORECAST_DATE.date()}",
    xaxis_title="Hour Ending", yaxis_title="MW/hr",
    xaxis=dict(dtick=1),
    height=450,
    template="plotly_dark",
)
fig.show()

## 6. RTO Load Profile + Ramp Overlay

Absolute load (MW) on primary y-axis, ramp rate (MW/hr) on secondary y-axis.

In [7]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

dfr = df_da.sort_values("hour_ending")

# Load profile (primary y)
fig.add_trace(
    go.Scatter(
        x=dfr["hour_ending"], y=dfr["forecast_load_mw"],
        name="Load", line=dict(color="#636EFA", width=2.5),
        mode="lines+markers", marker=dict(size=5),
        hovertemplate="HE%{x}<br>Load: %{y:,.0f} MW<extra></extra>",
    ),
    secondary_y=False,
)

# Ramp rate (secondary y)
dfr_ramp = dfr.dropna(subset=["ramp_mw"])
ramp_colors = ["rgba(44, 160, 44, 0.7)" if v >= 0 else "rgba(214, 39, 40, 0.7)" for v in dfr_ramp["ramp_mw"]]

fig.add_trace(
    go.Bar(
        x=dfr_ramp["hour_ending"], y=dfr_ramp["ramp_mw"],
        name="Ramp", marker_color=ramp_colors, opacity=0.6,
        hovertemplate="HE%{x}<br>Ramp: %{y:+,.0f} MW/hr<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_yaxes(title_text="Load (MW)", secondary_y=False)
fig.update_yaxes(title_text="Ramp (MW/hr)", secondary_y=True)
fig.add_hline(y=0, line_color="gray", line_width=0.5, secondary_y=True)
fig.update_layout(
    title=f"RTO — Load Profile + Ramp Overlay — {FORECAST_DATE.date()}",
    xaxis_title="Hour Ending", xaxis=dict(dtick=1),
    height=500, template="plotly_dark",
)
fig.show()

## 7. RTO Max Ramp Summary

In [8]:
dfr = df_da[df_da["ramp_mw"].notna()]

idx_max = dfr["ramp_mw"].idxmax()
row_max = dfr.loc[idx_max]
idx_min = dfr["ramp_mw"].idxmin()
row_min = dfr.loc[idx_min]

summary_data = {
    "Metric": [
        "Max Ramp (MW)", "Max Ramp HE", "Max Ramp Load (MW)",
        "Min Ramp (MW)", "Min Ramp HE", "Min Ramp Load (MW)",
        "Avg Abs Ramp (MW)", "Peak Load (MW)", "Min Load (MW)",
    ],
    "Value": [
        f"{row_max['ramp_mw']:+,.0f}", int(row_max["hour_ending"]), f"{row_max['forecast_load_mw']:,.0f}",
        f"{row_min['ramp_mw']:+,.0f}", int(row_min["hour_ending"]), f"{row_min['forecast_load_mw']:,.0f}",
        f"{dfr['ramp_mw'].abs().mean():,.0f}",
        f"{dfr['forecast_load_mw'].max():,.0f}", f"{dfr['forecast_load_mw'].min():,.0f}",
    ],
}

df_summary = pd.DataFrame(summary_data)

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=["Metric", "Value"],
        fill_color="#1f2937",
        font=dict(color="white", size=13),
        align="left",
    ),
    cells=dict(
        values=[df_summary["Metric"], df_summary["Value"]],
        fill_color="#111827",
        font=dict(color="white", size=12),
        align="left",
    ),
)])
fig_table.update_layout(
    title=f"RTO — Max Ramp Summary — DA Forecast {FORECAST_DATE.date()}",
    template="plotly_dark", height=350,
    margin=dict(l=20, r=20, t=50, b=20),
)
fig_table.show()

## 8. Historical Ramp Comparison — RTO Forecast vs Actual

Compare DA forecasted ramps vs RT metered actual ramps for the previous 3 days.

In [9]:
if RT_AVAILABLE and df_da_hist is not None and len(df_da_hist) > 0:
    df_rt["date_str"] = df_rt["date"].astype(str)
    df_da_hist["date_str"] = df_da_hist["forecast_date"].astype(str)

    df_compare = pd.merge(
        df_da_hist[["date_str", "hour_ending", "forecast_load_mw", "ramp_mw"]],
        df_rt[["date_str", "hour_ending", "rt_load_mw", "ramp_mw"]],
        on=["date_str", "hour_ending"],
        how="inner",
        suffixes=("_da", "_rt"),
    )
    df_compare["ramp_error"] = df_compare["ramp_mw_da"] - df_compare["ramp_mw_rt"]

    logger.info(f"Ramp comparison: {len(df_compare):,} matched rows")

    # ── Plot: one subplot row per date ──
    hist_dates = sorted(df_compare["date_str"].unique())

    fig = make_subplots(
        rows=len(hist_dates), cols=1,
        shared_xaxes=True,
        subplot_titles=[f"RTO — {d}" for d in hist_dates],
        vertical_spacing=0.08,
    )

    for i, date in enumerate(hist_dates, 1):
        dfd = df_compare[df_compare["date_str"] == date].sort_values("hour_ending")

        fig.add_trace(go.Bar(
            x=dfd["hour_ending"], y=dfd["ramp_mw_da"],
            name="DA Forecast Ramp", marker_color="rgba(99, 110, 250, 0.7)",
            showlegend=(i == 1), legendgroup="da",
            hovertemplate="HE%{x}<br>DA Ramp: %{y:+,.0f} MW<extra></extra>",
        ), row=i, col=1)

        fig.add_trace(go.Scatter(
            x=dfd["hour_ending"], y=dfd["ramp_mw_rt"],
            name="RT Actual Ramp", mode="lines+markers",
            line=dict(color="#FFA15A", width=2), marker=dict(size=5),
            showlegend=(i == 1), legendgroup="rt",
            hovertemplate="HE%{x}<br>RT Ramp: %{y:+,.0f} MW<extra></extra>",
        ), row=i, col=1)

        fig.update_yaxes(title_text="MW/hr", row=i, col=1)
        fig.add_hline(y=0, line_color="gray", line_width=0.5, row=i, col=1)

    fig.update_xaxes(title_text="Hour Ending", dtick=1, row=len(hist_dates), col=1)
    fig.update_layout(
        height=300 * len(hist_dates), template="plotly_dark",
        title_text=f"RTO — DA Forecast vs RT Actual Ramps (Previous {HISTORY_DAYS} Days)",
    )
    fig.show()

    # ── Error Summary ──
    error_stats = df_compare["ramp_error"].dropna()
    print(f"\nRTO Ramp Forecast Error Summary (DA - RT, MW/hr):")
    print(f"  MAE:               {error_stats.abs().mean():>8,.0f} MW")
    print(f"  RMSE:              {np.sqrt((error_stats**2).mean()):>8,.0f} MW")
    print(f"  Mean Bias:         {error_stats.mean():>+8,.0f} MW")
    print(f"  Max Overforecast:  {error_stats.max():>+8,.0f} MW")
    print(f"  Max Underforecast: {error_stats.min():>+8,.0f} MW")

else:
    print("RT metered data not available — skipping historical ramp comparison.")

INFO:ramp_analysis:Ramp comparison: 47 matched rows



RTO Ramp Forecast Error Summary (DA - RT, MW/hr):
  MAE:                    491 MW
  RMSE:                   758 MW
  Mean Bias:               -7 MW
  Max Overforecast:    +1,151 MW
  Max Underforecast:   -2,603 MW
